In [183]:
import os
import pretty_midi
from collections import defaultdict
import pretty_midi
from music21 import chord
from collections import defaultdict
import torch
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from music21 import stream, note
from torchinfo import summary
import os
import torch
import pretty_midi
import re
import difflib
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

In [184]:
# Leemos los archivos MIDI de un directorio específico
# y los separamos en dos conjuntos: X (simples) e Y (enriquecidos).

midi_dir = "MIDI_TRAIN"

X_dataset = []
Y_dataset = []

for file_name in os.listdir(midi_dir):
    index = file_name.split("_")[0]
    if file_name.startswith(index + "_1_pf_org"):
        X_dataset.append((int(index), os.path.join(midi_dir, file_name)))
    elif file_name.startswith(index + "_2_pf_rhm"):
        Y_dataset.append((int(index), os.path.join(midi_dir, file_name)))

# Ordenamos por índice
X_dataset.sort(key=lambda x: x[0])
Y_dataset.sort(key=lambda x: x[0])

# Nos quedamos solo con los paths
X_dataset = [x[1] for x in X_dataset]
Y_dataset = [y[1] for y in Y_dataset]

print("X_dataset ordenado:", X_dataset)
print("Y_dataset ordenado:", Y_dataset)


X_dataset ordenado: ['MIDI_TRAIN/01_1_pf_org.mid', 'MIDI_TRAIN/02_1_pf_org.mid', 'MIDI_TRAIN/03_1_pf_org.mid', 'MIDI_TRAIN/04_1_pf_org.mid', 'MIDI_TRAIN/05_1_pf_org.mid', 'MIDI_TRAIN/06_1_pf_org.mid', 'MIDI_TRAIN/07_1_pf_org.mid', 'MIDI_TRAIN/08_1_pf_org.mid', 'MIDI_TRAIN/09_1_pf_org.mid', 'MIDI_TRAIN/10_1_pf_org.mid', 'MIDI_TRAIN/11_1_pf_org.mid', 'MIDI_TRAIN/12_1_pf_org.mid', 'MIDI_TRAIN/13_1_pf_org.mid', 'MIDI_TRAIN/14_1_pf_org.mid', 'MIDI_TRAIN/15_1_pf_org.mid', 'MIDI_TRAIN/16_1_pf_org.mid', 'MIDI_TRAIN/17_1_pf_org.mid', 'MIDI_TRAIN/18_1_pf_org.mid', 'MIDI_TRAIN/19_1_pf_org.mid', 'MIDI_TRAIN/20_1_pf_org.mid', 'MIDI_TRAIN/21_1_pf_org.mid', 'MIDI_TRAIN/22_1_pf_org.mid', 'MIDI_TRAIN/23_1_pf_org.mid', 'MIDI_TRAIN/24_1_pf_org.mid', 'MIDI_TRAIN/25_1_pf_org.mid', 'MIDI_TRAIN/26_1_pf_org.mid', 'MIDI_TRAIN/27_1_pf_org.mid', 'MIDI_TRAIN/28_1_pf_org.mid', 'MIDI_TRAIN/29_1_pf_org.mid', 'MIDI_TRAIN/30_1_pf_org.mid', 'MIDI_TRAIN/31_1_pf_org.mid', 'MIDI_TRAIN/32_1_pf_org.mid', 'MIDI_TRAIN/33_1_pf

In [185]:
# Función para convertir MIDI a acordes normalizados
# Detecta inversiones y acordes complejos, y agrupa notas por tiempo.
# Utiliza una tolerancia para redondear los tiempos de las notas.
# También descarta grupos de notas con menos de un número mínimo especificado.
# Utiliza pretty_midi para manejar los archivos MIDI y music21 para el manejo de acordes.
# La función devuelve una lista de conjuntos de clases de pitch (pitch class sets) para cada acorde detectado.
# Los acordes se representan como listas de números enteros que indican las notas en la escala cromática (0-11).

def midi_to_chords(midi_path, tolerance=0.1, min_notes=2):
    pm = pretty_midi.PrettyMIDI(midi_path)
    all_notes = []

    for instrument in pm.instruments:
        if instrument.is_drum:
            continue
        all_notes.extend(instrument.notes)

    notes_by_time = defaultdict(list)
    for note in all_notes:
        time_key = round(note.start / tolerance) * tolerance
        notes_by_time[time_key].append(note.pitch)

    chords = []

    for time in sorted(notes_by_time.keys()):
        note_group = notes_by_time[time]
        if len(note_group) < min_notes:
            continue 
        try:
            m21_chord = chord.Chord(note_group)
            pc_set = m21_chord.normalOrder
        except:
            pc_set = sorted(set(n % 12 for n in note_group))
        chords.append(pc_set)

    return chords


X_dataset = [midi_to_chords(path) for path in X_dataset]
Y_dataset = [midi_to_chords(path) for path in Y_dataset]

print("Count of X_dataset:", len(X_dataset))
print("Count of Y_dataset:", len(Y_dataset))
print("Processed Input (X):",  X_dataset)
print("Processed Output (Y):", Y_dataset)

Count of X_dataset: 50
Count of Y_dataset: 50
Processed Input (X): [[[0, 4, 7], [9, 0, 4], [2, 5, 9], [7, 11, 2]], [[9, 0, 4], [2, 5, 9], [4, 8, 11], [1, 4, 7, 9], [2, 5, 9], [4, 8, 11], [9, 0, 4]], [[0, 4, 7], [9, 0, 4], [2, 5, 9], [7, 11, 2]], [[0, 4, 7], [9, 0, 4], [2, 5, 9], [7, 11, 2]], [[0, 4, 7], [9, 0, 4], [2, 5, 9], [7, 11, 2]], [[0, 4, 7], [9, 0, 4], [2, 5, 9], [7, 11, 2]], [[0, 4, 7], [9, 0, 4], [2, 5, 9], [7, 11, 2]], [[0, 4, 7], [9, 0, 4], [2, 5, 9], [7, 11, 2]], [[9, 0, 4], [2, 5, 9], [4, 8, 11], [1, 4, 7, 9], [2, 5, 9], [4, 8, 11], [9, 0, 4]], [[0, 4, 7], [9, 0, 4], [2, 5, 9], [7, 11, 2]], [[5, 9, 0], [7, 11, 2], [0, 4, 7], [9, 0, 4]], [[2, 5, 9], [2, 5, 9], [7, 11, 2], [7, 11, 2]], [[2, 5, 9], [2, 5, 9], [7, 11, 2], [7, 11, 2]], [[2, 5, 9], [7, 11, 2], [0, 4, 7], [3, 6, 9, 11], [8, 11, 2, 4]], [[5, 9, 0], [0, 4, 7], [5, 9, 0], [0, 4, 7], [9, 0, 2, 5], [11, 2, 4, 7], [5, 9, 0], [11, 2, 5, 7]], [[2, 5, 9], [11, 2, 5, 7], [0, 4, 7], [1, 4, 7, 9]], [[5, 9, 0], [0, 4, 7], [2

In [186]:
# === DATA AUGMENTATION POR TRANSPOSICIÓN ===
X_augmented = []
Y_augmented = []

def transpose_progression(prog, semitones):
    return [[(n + semitones) % 12 for n in chord] for chord in prog]

for x_prog, y_prog in zip(X_dataset, Y_dataset):
    for shift in range(12):  # transposición de 0 a 11 semitonos
        x_trans = transpose_progression(x_prog, shift)
        y_trans = transpose_progression(y_prog, shift)
        X_augmented.append(x_trans)
        Y_augmented.append(y_trans)

X_dataset = X_augmented
Y_dataset = Y_augmented

print("Count of X_dataset:", len(X_dataset))
print("Count of Y_dataset:", len(Y_dataset))
print("Processed Input (X):",  X_dataset)
print("Processed Output (Y):", Y_dataset)

Count of X_dataset: 600
Count of Y_dataset: 600
Processed Input (X): [[[0, 4, 7], [9, 0, 4], [2, 5, 9], [7, 11, 2]], [[1, 5, 8], [10, 1, 5], [3, 6, 10], [8, 0, 3]], [[2, 6, 9], [11, 2, 6], [4, 7, 11], [9, 1, 4]], [[3, 7, 10], [0, 3, 7], [5, 8, 0], [10, 2, 5]], [[4, 8, 11], [1, 4, 8], [6, 9, 1], [11, 3, 6]], [[5, 9, 0], [2, 5, 9], [7, 10, 2], [0, 4, 7]], [[6, 10, 1], [3, 6, 10], [8, 11, 3], [1, 5, 8]], [[7, 11, 2], [4, 7, 11], [9, 0, 4], [2, 6, 9]], [[8, 0, 3], [5, 8, 0], [10, 1, 5], [3, 7, 10]], [[9, 1, 4], [6, 9, 1], [11, 2, 6], [4, 8, 11]], [[10, 2, 5], [7, 10, 2], [0, 3, 7], [5, 9, 0]], [[11, 3, 6], [8, 11, 3], [1, 4, 8], [6, 10, 1]], [[9, 0, 4], [2, 5, 9], [4, 8, 11], [1, 4, 7, 9], [2, 5, 9], [4, 8, 11], [9, 0, 4]], [[10, 1, 5], [3, 6, 10], [5, 9, 0], [2, 5, 8, 10], [3, 6, 10], [5, 9, 0], [10, 1, 5]], [[11, 2, 6], [4, 7, 11], [6, 10, 1], [3, 6, 9, 11], [4, 7, 11], [6, 10, 1], [11, 2, 6]], [[0, 3, 7], [5, 8, 0], [7, 11, 2], [4, 7, 10, 0], [5, 8, 0], [7, 11, 2], [0, 3, 7]], [[1, 4, 8

In [187]:
# Función para convertir un acorde (lista de notas) a un token único.
# Los tokens se generan concatenando las notas del acorde, ordenadas y separadas por guiones.
def chord_to_token(chord):
    return '_'.join(map(str, sorted(chord)))


# Crear un vocabulario de tokens a partir de los acordes en los datasets X e Y.
# Cada token es una cadena que representa un acorde, y se asigna un ID único a cada token.
# Se incluyen tokens especiales para padding, inicio de secuencia y fin de secuencia.

token_to_id = {'<pad>': 0, '<sos>': 1, '<eos>': 2} 
id_to_token = {0: '<pad>', 1: '<sos>', 2: '<eos>'}
current_id = 3
for progression in X_dataset + Y_dataset:
    for chord in progression:
        token = chord_to_token(chord)
        if token not in token_to_id:
            token_to_id[token] = current_id
            id_to_token[current_id] = token
            current_id += 1
vocab_size = len(token_to_id)



# Función para manejar casos donde un acorde no se encuentra en el vocabulario.
# Utiliza difflib para encontrar el token más parecido en caso de que el acorde no
# esté presente en el vocabulario.

def fallback_token_str(chord_str, token_to_id):
    matches = difflib.get_close_matches(chord_str, token_to_id.keys(), n=1)
    return token_to_id[matches[0]] if matches else token_to_id.get('<pad>', 0)


# Función para convertir una progresión de acordes a una secuencia de tokens.
# Utiliza la función chord_to_token para convertir cada acorde a un token,
# y luego busca el ID correspondiente en el vocabulario.
# Si un token no se encuentra, utiliza la función fallback_token_str para encontrar el token más parecido.
# Devuelve una lista de IDs de tokens que representan la progresión de acordes.
# Los tokens se convierten a tensores de PyTorch para su uso en redes neuronales.
def progression_to_token_seq(prog):
    tokens = []
    for chord in prog:
        token = chord_to_token(chord)
        if token in token_to_id:
            tokens.append(token_to_id[token])
        else:
            # Buscar token más parecido
            fallback = fallback_token_str(token, token_to_id)
            tokens.append(fallback)
    return tokens



# Convertir las progresiones de acordes en secuencias de tokens.
# Cada secuencia de tokens se convierte en un tensor de PyTorch.
X_data = []
Y_data = []

for x_prog, y_prog in zip(X_dataset, Y_dataset):
    x_tokens = progression_to_token_seq(x_prog)
    y_tokens = [token_to_id['<sos>']] + progression_to_token_seq(y_prog) + [token_to_id['<eos>']]
    
    X_data.append(torch.tensor(x_tokens, dtype=torch.long))
    Y_data.append(torch.tensor(y_tokens, dtype=torch.long))


# Guardar los datos procesados y el vocabulario en archivos para su uso posterior.
with open("token_to_id.pkl", "wb") as f:
    pickle.dump(token_to_id, f)

with open("id_to_token.pkl", "wb") as f:
    pickle.dump(id_to_token, f)



# Definir un dataset personalizado para manejar pares de progresiones de acordes.
# El dataset devuelve pares de tensores (X, Y) donde X es la secuencia
# de tokens de la progresión de acordes simple y Y es la secuencia de tokens
# de la progresión enriquecida. Utiliza pad_sequence para manejar el padding
# de las secuencias de diferentes longitudes en un batch.

class ChordPairDataset(Dataset):
    def __init__(self, X_data, Y_data):
        self.X_data = X_data
        self.Y_data = Y_data

    def __len__(self):
        return len(self.X_data)

    def __getitem__(self, idx):
        return self.X_data[idx], self.Y_data[idx]
    
    def collate_fn(batch):
        X_batch, Y_batch = zip(*batch)
        X_padded = pad_sequence(X_batch, batch_first=True, padding_value=0)
        Y_padded = pad_sequence(Y_batch, batch_first=True, padding_value=0)
        return X_padded, Y_padded
    
collate_fn = ChordPairDataset.collate_fn


# Crear el dataset y el DataLoader 
dataset = ChordPairDataset(X_data, Y_data)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=
                        collate_fn)


# Ejemplo de uso del DataLoader
for X_batch, Y_batch in dataloader:
    print("X batch shape:", X_batch.shape)
    print("Y batch shape:", Y_batch.shape)
    break
print("Total batches:", len(dataloader))



from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

# División en entrenamiento y validación (ejemplo: 80% train, 20% val)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_data, Y_data, test_size=0.2, random_state=42)

# Crear datasets de entrenamiento y validación
train_dataset = ChordPairDataset(X_train, Y_train)
val_dataset = ChordPairDataset(X_val, Y_val)

# Crear DataLoaders para entrenamiento y validación
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)


# Función para convertir una secuencia de tokens a una lista de acordes.
# Utiliza el diccionario id_to_token para mapear los IDs de tokens a sus representaciones de acordes.
# Descarta tokens especiales como padding, inicio y fin de secuencia.
# Devuelve una lista de acordes, donde cada acorde es una lista de números enteros
# que representan las notas en la escala cromática (0-11).
def token_seq_to_chords(token_seq, id_to_token):
    chords = []
    for idx in token_seq:
        token = id_to_token.get(idx, '<unk>')
        if token in ['<pad>', '<sos>', '<eos>']:
            continue
        chord = list(map(int, token.split('_')))
        chords.append(chord)
    return chords


X batch shape: torch.Size([8, 8])
Y batch shape: torch.Size([8, 10])
Total batches: 75


In [188]:
# Definir el modelo CVAE con LSTM.
# El modelo consta de un codificador LSTM que genera un vector latente a partir de
# una secuencia de entrada, y un decodificador LSTM que genera una secuencia
# de salida a partir del vector latente y una secuencia de entrada (con teacher forcing).

class CVAE_LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, vocab_size, embedding_dim, num_layers=2, dropout=0.2):
        super(CVAE_LSTM, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.vocab_size = vocab_size

        # Encoder
        self.encoder_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.encoder_lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # Decoder
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.decoder_lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.latent_to_hidden = nn.Linear(latent_dim, hidden_dim * num_layers)
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def encode(self, x):
        x = self.encoder_embedding(x)  # [batch, seq_len, emb_dim]
        _, (h_n, _) = self.encoder_lstm(x)
        h_last = h_n[-1]
        mu = self.fc_mu(h_last)
        logvar = self.fc_logvar(h_last)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, y_seq, teacher_forcing_ratio=0.4):
        batch_size, seq_len = y_seq.size()
        embedding = self.embedding

        hidden_state = torch.tanh(self.latent_to_hidden(z))
        hidden = hidden_state.view(self.decoder_lstm.num_layers, batch_size, self.hidden_dim)
        cell = torch.zeros_like(hidden).to(hidden.device)

        inputs = y_seq[:, 0]
        outputs = []

        for t in range(1, seq_len):
            input_embed = embedding(inputs).unsqueeze(1)
            output, (hidden, cell) = self.decoder_lstm(input_embed, (hidden, cell))
            output_logits = self.output_layer(output.squeeze(1))
            outputs.append(output_logits)

            teacher_force = (torch.rand(1).item() < teacher_forcing_ratio)
            top1 = output_logits.argmax(1)
            inputs = y_seq[:, t] if teacher_force else top1

        outputs = torch.stack(outputs, dim=1)
        return outputs

    def forward(self, x, y_seq=None, teacher_forcing_ratio=0.8):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        if y_seq is not None:
            y_hat = self.decode(z, y_seq, teacher_forcing_ratio)
            return y_hat, mu, logvar
        else:
            # Modo generación sin y_seq
            y_hat = self.generate(z, max_len=10)  # max_len puede ser parámetro externo
            return y_hat, mu, logvar

    # Método para generar secuencias a partir del vector latente z.
    def generate(self, z, max_len=10, start_token_id=None, eos_token_id=None):
        batch_size = z.size(0)
        hidden_state = torch.tanh(self.latent_to_hidden(z))
        hidden = hidden_state.view(self.decoder_lstm.num_layers, batch_size, self.hidden_dim)
        cell = torch.zeros_like(hidden).to(hidden.device)

        inputs = torch.full((batch_size,), start_token_id, dtype=torch.long).to(z.device)
        generated_tokens = []

        for _ in range(max_len):
            input_embed = self.embedding(inputs).unsqueeze(1)
            output, (hidden, cell) = self.decoder_lstm(input_embed, (hidden, cell))
            output_logits = self.output_layer(output.squeeze(1))  # [batch, vocab_size]

            probs = torch.softmax(output_logits, dim=-1)
            inputs = torch.multinomial(probs, num_samples=1).squeeze(1)

            generated_tokens.append(inputs)

            if eos_token_id is not None:
                if (inputs == eos_token_id).all():
                    break

        generated_tokens = torch.stack(generated_tokens, dim=1)  # [batch, seq_len]
        return generated_tokens


# Función auxiliar para obtener escala mod12 desde notas MIDI (sin cambios)
def detectar_escala(notas_midi):
    s = stream.Stream()
    for n in notas_midi:
        s.append(note.Note(n))
    k = s.analyze('key')
    escala = k.getScale().getPitches()
    escala_mod12 = sorted({p.pitchClass for p in escala})
    return escala_mod12

# Convertir tokens a root notes.
def tokens_to_root_notes(tokens):
    root_notes = []
    for token in tokens:
        chord = id_to_token[token]
        if chord in ['<sos>', '<eos>', '<pad>']:
            continue
        try:
            notas = [int(x) for x in chord.split('_')]
            root_note = min(notas)
            root_notes.append(root_note)
        except ValueError:
            continue
    return root_notes


# Loss diferenciable para coherencia tonal.
# Penaliza acordes que no están en la escala tonal esperada.
def loss_coherencia_tonal_soft(y_hat_step, root_note, escala_mod12):
    probs = F.softmax(y_hat_step, dim=-1)  # [batch, vocab_size]
    fuera_escala = torch.zeros(probs.size(-1), device=probs.device)
    
    for idx in range(probs.size(-1)):
        chord = id_to_token[idx]
        if chord in ['<sos>', '<eos>', '<pad>']:
            fuera_escala[idx] = 0  # No penalizar tokens especiales
        else:
            try:
                notas = [root_note + int(x) for x in chord.split('_')]
                # Penaliza cuántas notas del acorde están fuera de escala
                count_fuera = sum(1 for n in notas if n % 12 not in escala_mod12)
                fuera_escala[idx] = count_fuera / len(notas)
            except:
                fuera_escala[idx] = 0

    # Penalización suave: suma probabilidades de acordes fuera de escala ponderado
    loss = torch.sum(probs * fuera_escala)
    return loss

# Loss diferenciable para movimiento armónico.
# Penaliza saltos grandes entre notas raíz de acordes consecutivos.
def loss_movimiento_armonico_soft(root_notes, device='cpu'):
    loss = 0.0
    for i in range(1, len(root_notes)):
        salto = abs(root_notes[i] - root_notes[i-1])
        salto_tensor = torch.tensor([salto - 7.0], device=device)  # convertir a tensor
        loss += F.relu(salto_tensor).item()  # convertir de nuevo a escalar con .item()
    return loss / max(1, len(root_notes)-1)


# Loss diferenciable para tensiones.
# Penaliza acordes con menos de 4 notas (menos tensiones).
def loss_tensiones_soft(y_hat_step):
    probs = F.softmax(y_hat_step, dim=-1)
    penalizacion = torch.zeros(probs.size(-1), device=probs.device)

    for idx in range(probs.size(-1)):
        chord = id_to_token[idx]
        if chord in ['<sos>', '<eos>', '<pad>']:
            penalizacion[idx] = 0
        else:
            try:
                num_notas = len(chord.split('_'))
                penalizacion[idx] = 1 if num_notas < 4 else 0
            except:
                penalizacion[idx] = 0

    loss = torch.sum(probs * penalizacion)
    return loss


# Función de pérdida para el CVAE.
# Calcula la pérdida de reconstrucción (cross-entropy) y la pérdida KL-divergencia.
# La pérdida total es la suma de ambas, ponderada por un factor beta.
def cvae_loss(y_hat, y, mu, logvar, beta=0.1, free_bits=1.0):
    y_target = y[:, 1:]
    recon_loss = F.cross_entropy(
        y_hat.reshape(-1, y_hat.size(-1)),
        y_target.reshape(-1),
        reduction='sum'
    )
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    kl_loss = torch.clamp(kl_loss, min=free_bits)
    return recon_loss + beta * kl_loss, recon_loss, kl_loss


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Definir el modelo CVAE con LSTM.
# El modelo utiliza embeddings para los tokens, LSTM para codificación y decodificación,
# y una capa lineal para mapear el estado oculto a logits de salida.
model = CVAE_LSTM(
    input_dim=32, 
    hidden_dim=16,
    embedding_dim=64,
    latent_dim=12,
    num_layers=2, 
    dropout=0.4,
    vocab_size=vocab_size,
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [189]:
# Parámetros de entrenamiento
num_epochs = 100
coherencia_tonal_weight = 0.8
movimiento_armonico_weight = 0.1
tensiones_weight = 0.1

# Factor de ponderación para la pérdida musical
lambda_music = 0.3


best_val_loss = float("inf")
patience = 5
counter = 0

# KL Annealing (incremento progresivo del peso KL)
beta_max = 1.0       # Peso máximo para KL
warmup_epochs = 50    # Número de epochs para llegar a beta_max

def get_beta(epoch, cycle_length=50, beta_max=1.0):
    progress = (epoch % cycle_length) / cycle_length
    return beta_max * min(1.0, progress)


# Entrenar el modelo CVAE con LSTM.
for epoch in range(num_epochs):
    beta = get_beta(epoch)
    model.train()
    total_loss_train = 0
    total_recon_train = 0
    total_kl_train = 0
    total_music_train = 0

    for X_batch, Y_batch in train_loader:
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        Y_hat, mu, logvar = model(X_batch.long(), Y_batch)

        loss_base, recon, kl = cvae_loss(Y_hat, Y_batch, mu, logvar, beta=beta)


        batch_size, seq_len_minus1, vocab_size = Y_hat.shape

        batch_musical_loss = 0
        for i in range(batch_size):
            tokens_i = X_batch[i].cpu().numpy().tolist()
            notas_i = tokens_to_root_notes(tokens_i)
            escala_i = detectar_escala(notas_i)
            root_notes_gen = notas_i

            loss_coherencia = 0
            loss_tensiones = 0
            for t in range(seq_len_minus1):
                y_hat_step = Y_hat[i, t]
                root_note = root_notes_gen[t] if t < len(root_notes_gen) else 60
                loss_coherencia += loss_coherencia_tonal_soft(y_hat_step.unsqueeze(0), root_note, escala_i)
                loss_tensiones += loss_tensiones_soft(y_hat_step.unsqueeze(0))

            loss_coherencia /= seq_len_minus1
            loss_tensiones /= seq_len_minus1

            loss_armonico = loss_movimiento_armonico_soft(root_notes_gen, device=device)

            batch_musical_loss += (
                coherencia_tonal_weight * loss_coherencia +
                movimiento_armonico_weight * loss_armonico +
                tensiones_weight * loss_tensiones
            )
        
        batch_musical_loss /= batch_size

        loss = loss_base + lambda_music * batch_musical_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss_train += loss.item()
        total_recon_train += recon.item()
        total_kl_train += kl.item()
        total_music_train += batch_musical_loss.item()

    # Validación
    model.eval()
    total_loss_val = 0
    total_recon_val = 0
    total_kl_val = 0
    total_music_val = 0

    with torch.no_grad():
        for X_batch, Y_batch in val_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            Y_hat, mu, logvar = model(X_batch.long(), Y_batch)

            loss_base, recon, kl = cvae_loss(Y_hat, Y_batch, mu, logvar, beta=beta)

            batch_size, seq_len_minus1, vocab_size = Y_hat.shape

            batch_musical_loss = 0
            for i in range(batch_size):
                tokens_i = X_batch[i].cpu().numpy().tolist()
                notas_i = tokens_to_root_notes(tokens_i)
                escala_i = detectar_escala(notas_i)
                root_notes_gen = notas_i

                loss_coherencia = 0
                loss_tensiones = 0
                for t in range(seq_len_minus1):
                    y_hat_step = Y_hat[i, t]
                    root_note = root_notes_gen[t] if t < len(root_notes_gen) else 60
                    loss_coherencia += loss_coherencia_tonal_soft(y_hat_step.unsqueeze(0), root_note, escala_i)
                    loss_tensiones += loss_tensiones_soft(y_hat_step.unsqueeze(0))

                loss_coherencia /= seq_len_minus1
                loss_tensiones /= seq_len_minus1

                loss_armonico = loss_movimiento_armonico_soft(root_notes_gen, device=device)

                batch_musical_loss += (
                    coherencia_tonal_weight * loss_coherencia +
                    movimiento_armonico_weight * loss_armonico +
                    tensiones_weight * loss_tensiones
                )

            batch_musical_loss /= batch_size

            loss = loss_base + lambda_music * batch_musical_loss

            total_loss_val += loss.item()
            total_recon_val += recon.item()
            total_kl_val += kl.item()
            total_music_val += batch_musical_loss.item()

    # Mostrar métricas
    print(f"[Epoch {epoch+1}/{num_epochs}]")
    print(f" Train Loss: {total_loss_train:.2f} | Recon: {total_recon_train:.2f} | KL: {total_kl_train:.2f} | Music: {total_music_train:.2f}")
    print(f" Val   Loss: {total_loss_val:.2f} | Recon: {total_recon_val:.2f} | KL: {total_kl_val:.2f} | Music: {total_music_val:.2f}")
    if total_loss_val < best_val_loss:
        best_val_loss = total_loss_val
        counter = 0
        torch.save(model.state_dict(), f"modelo_best.pth")
    else:
        counter += 1
        if counter >= patience:
            print(f"⛔ Early stopping en epoch {epoch+1}")
            break

[Epoch 1/100]
 Train Loss: 26322.40 | Recon: 26316.15 | KL: 84.51 | Music: 20.83
 Val   Loss: 7918.98 | Recon: 7917.42 | KL: 20.13 | Music: 5.21
[Epoch 2/100]
 Train Loss: 26038.05 | Recon: 26030.28 | KL: 76.02 | Music: 20.83
 Val   Loss: 7883.17 | Recon: 7881.28 | KL: 16.29 | Music: 5.21
[Epoch 3/100]
 Train Loss: 26276.35 | Recon: 26267.59 | KL: 62.82 | Music: 20.83
 Val   Loss: 7836.79 | Recon: 7834.63 | KL: 15.03 | Music: 5.21
[Epoch 4/100]
 Train Loss: 26008.44 | Recon: 25998.54 | KL: 60.78 | Music: 20.83
 Val   Loss: 7775.55 | Recon: 7773.09 | KL: 15.00 | Music: 5.21
[Epoch 5/100]
 Train Loss: 25919.12 | Recon: 25908.05 | KL: 60.26 | Music: 20.83
 Val   Loss: 7676.06 | Recon: 7673.30 | KL: 15.00 | Music: 5.21
[Epoch 6/100]
 Train Loss: 25393.47 | Recon: 25381.21 | KL: 60.11 | Music: 20.82
 Val   Loss: 7568.80 | Recon: 7565.74 | KL: 15.00 | Music: 5.20
[Epoch 7/100]
 Train Loss: 25355.07 | Recon: 25341.60 | KL: 60.24 | Music: 20.82
 Val   Loss: 7418.36 | Recon: 7415.00 | KL: 15.00

In [192]:
batch_size = 8
vocab_size = len(token_to_id)
seq_len = 4

dummy_input = torch.randint(0, vocab_size, (batch_size, seq_len), dtype=torch.long).to(device)
dummy_target = torch.randint(0, vocab_size, (batch_size, seq_len), dtype=torch.long).to(device)

summary(model, input_data=(dummy_input, dummy_target))

Layer (type:depth-idx)                   Output Shape              Param #
CVAE_LSTM                                [8, 3, 310]               --
├─Embedding: 1-1                         [8, 4, 64]                19,840
├─LSTM: 1-2                              [8, 4, 16]                7,424
├─Linear: 1-3                            [8, 12]                   204
├─Linear: 1-4                            [8, 12]                   204
├─Linear: 1-5                            [8, 32]                   416
├─Embedding: 1-6                         [8, 64]                   19,840
├─LSTM: 1-7                              [8, 1, 16]                7,424
├─Linear: 1-8                            [8, 310]                  5,270
├─Embedding: 1-9                         [8, 64]                   (recursive)
├─LSTM: 1-10                             [8, 1, 16]                (recursive)
├─Linear: 1-11                           [8, 310]                  (recursive)
├─Embedding: 1-12                     

In [1]:
import os
import re
import torch
import pretty_midi
import IPython.display as ipd

# --- Funciones útiles ---

def chord_to_set(chord_str):
    return set(int(pc) for pc in chord_str.split('_'))

def find_closest_chord(target_chord, vocab_chords, max_diff_threshold=3):
    target_set = chord_to_set(target_chord)
    min_diff = float('inf')
    closest = None
    for chord in vocab_chords:
        chord_set = chord_to_set(chord)
        diff = len(target_set.symmetric_difference(chord_set))
        if diff < min_diff:
            min_diff = diff
            closest = chord
    if min_diff <= max_diff_threshold:
        return closest
    else:
        return '0_4_7' if '0_4_7' in vocab_chords else list(vocab_chords)[0]

def midi_to_chords(midi_path, min_notes_per_chord=2):
    midi = pretty_midi.PrettyMIDI(midi_path)
    instrument = midi.instruments[0]
    time_notes = {}
    for note in instrument.notes:
        time_key = round(note.start, 2)
        if time_key not in time_notes:
            time_notes[time_key] = []
        time_notes[time_key].append(note.pitch)
    chords = []
    for time in sorted(time_notes.keys()):
        pitches = sorted(time_notes[time])
        if len(pitches) >= min_notes_per_chord:
            chord_str = '_'.join(str(p) for p in pitches)
            chords.append(chord_str)
    return chords

def chord_to_token(chord_str):
    notes = [int(p) for p in chord_str.split('_')]
    pitch_classes = sorted(set([n % 12 for n in notes]))
    return '_'.join(str(pc) for pc in pitch_classes)

def progression_to_token_seq(progression, token_to_id):
    tokens = []
    for chord in progression:
        tokens.append(token_to_id.get(chord, token_to_id['<unk>']))
    return [token_to_id['<sos>']] + tokens + [token_to_id['<eos>']]

def chords_to_notes(chord_str):
    return [int(n) for n in chord_str.split('_') if n.isdigit()]

def build_midi_from_chords(generated_chords, reference_midi_path, output_path):
    original_midi = pretty_midi.PrettyMIDI(reference_midi_path)
    original_duration = original_midi.get_end_time()
    tempo = original_midi.get_tempo_changes()[1][0]

    valid_chords = [chord for chord in generated_chords if chord not in ['<sos>', '<eos>', '<pad>']]
    num_chords = len(valid_chords)
    if num_chords == 0:
        raise ValueError("No hay acordes válidos para generar el MIDI.")

    chord_duration = original_duration / num_chords
    new_midi = pretty_midi.PrettyMIDI(initial_tempo=tempo)
    piano = pretty_midi.Instrument(program=0)

    time = 0.0
    for chord in valid_chords:
        notes_in_chord = chords_to_notes(chord)
        for n in notes_in_chord:
            pitch = 48 + n  # Base C3
            if 0 <= pitch <= 127:
                note = pretty_midi.Note(velocity=100, pitch=pitch, start=time, end=time + chord_duration)
                piano.notes.append(note)
        time += chord_duration

    new_midi.instruments.append(piano)
    new_midi.write(output_path)
    print(f"[✓] MIDI generado y guardado como {output_path}")

def mostrar_midi(path):
    if os.path.exists(path):
        return ipd.Audio(path)
    else:
        print(f"⚠️ El archivo {path} no existe.")
        return None


# --- Configuración inicial y vocabulario ---

# Asumiendo que ya tienes X_train, Y_train, X_pretrain, Y_pretrain como listas de tensores de tokens,
# y un diccionario id_to_token que mapea ID a token string.

# Construir vocabulario pitch-class a partir de datos
chord_set = set()
for progression in X_train + Y_train + X_pretrain + Y_pretrain:
    for token in progression:
        if isinstance(token, torch.Tensor):
            token = token.item()
        chord_str = id_to_token.get(token, '<unk>')
        if chord_str != '<unk>' and '_' in chord_str:
            pc_token = chord_to_token(chord_str)
            chord_set.add(pc_token)

chord_vocab = sorted(list(chord_set))
token_to_id = {'<pad>': 0, '<sos>':1, '<eos>':2, '<unk>':3}
for i, chord in enumerate(chord_vocab, start=4):
    token_to_id[chord] = i
id_to_token = {v:k for k,v in token_to_id.items()}


# --- Procesar archivo MIDI y generar progresión ---

nombre_archivo = "TEST1.mid"
input_path = os.path.join("MIDI_TEST", nombre_archivo)
base_name = os.path.splitext(nombre_archivo)[0]
new_base_name = re.sub(r'\d+$', base_name[-1], "GENERATED" + base_name[-1])
output_path = os.path.join("RESULTADOS_TEST", f"{new_base_name}.mid")

# Extraer acordes desde MIDI y normalizar a pitch class
x_input_chords = midi_to_chords(input_path)
x_input_chords = [chord_to_token(c) for c in x_input_chords]

# Reemplazar acordes fuera del vocabulario por el más cercano
corrected_chords = []
for chord in x_input_chords:
    if chord in token_to_id:
        corrected_chords.append(chord)
    else:
        closest = find_closest_chord(chord, chord_vocab)
        corrected_chords.append(closest)
        print(f"🔄 Acorde '{chord}' no en vocabulario, reemplazado por '{closest}'.")

print("\n🎵 Progresión original:", x_input_chords)
print("🎵 Progresión corregida:", corrected_chords)

x_input_tokens = progression_to_token_seq(corrected_chords, token_to_id)
x_input_tensor = torch.tensor(x_input_tokens, dtype=torch.long).unsqueeze(0).to(device)

# Generar con el modelo CVAE
model.eval()
with torch.no_grad():
    mu, logvar = model.encode(x_input_tensor)
    z = model.reparameterize(mu, logvar)
    seq_len = x_input_tensor.size(1)
    max_attempts = 3
    for attempt in range(max_attempts):
        generated = model.generate(
            z,
            max_len=seq_len,
            start_token_id=token_to_id['<sos>'],
            eos_token_id=token_to_id['<eos>']
        )
        if generated.size(1) >= seq_len:
            break

# Convertir tokens generados a acordes (filtrando tokens especiales)
generated_chords = [id_to_token.get(token.item(), '<unk>') for token in generated[0]]
print("\n🎶 Progresión generada:", generated_chords)

# Guardar MIDI generado
os.makedirs("RESULTADOS_TEST", exist_ok=True)
print(f"\n🎧 Generando archivo MIDI en: {output_path}")
build_midi_from_chords(generated_chords, reference_midi_path=input_path, output_path=output_path)

# Mostrar MIDI en notebook
ipd.display(mostrar_midi(output_path))


NameError: name 'X_train' is not defined